In [1]:
import math
import random
import numpy as np

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator


class OQMSA:
    """
    Optimized Quantum Minimum Searching Algorithm (OQMSA)

    Based on the structure of Algorithm 1 and Algorithm 2
    in the supplied paper.

    Important:
        N  = number of actual database items
        n  = number of qubits needed to encode the largest value
        Ne = 2^n = size of the quantum search space
    """

    def __init__(self, dataset, seed=None):

        # ---------------------------------------------------------
        # 1. Check dataset
        # ---------------------------------------------------------
        if dataset is None or len(dataset) == 0:
            raise ValueError("Dataset cannot be empty.")

        if any(int(x) != x or x < 0 for x in dataset):
            raise ValueError(
                "This implementation expects non-negative integer data."
            )

        self.dataset = [int(x) for x in dataset]

        # ---------------------------------------------------------
        # 2. Number of actual data items
        # ---------------------------------------------------------
        self.N = len(self.dataset)

        # ---------------------------------------------------------
        # 3. Number of qubits
        #
        # We need enough qubits to represent max(dataset).
        #
        # Example:
        # max(dataset) = 45
        # binary(45) = 101101
        # therefore n = 6
        # ---------------------------------------------------------
        self.max_value = max(self.dataset)

        self.n = max(
            1,
            math.ceil(math.log2(self.max_value + 1))
        )

        # ---------------------------------------------------------
        # 4. Quantum search-space size
        # ---------------------------------------------------------
        self.Ne = 2 ** self.n

        # ---------------------------------------------------------
        # 5. Random generator
        # ---------------------------------------------------------
        self.rng = random.Random(seed)

        # ---------------------------------------------------------
        # 6. Simulation backend
        # ---------------------------------------------------------
        self.backend = AerSimulator()

        # ---------------------------------------------------------
        # 7. History
        # ---------------------------------------------------------
        self.history = []

        # ---------------------------------------------------------
        # Print configuration
        # ---------------------------------------------------------
        print("========================================")
        print("OQMSA CONFIGURATION")
        print("========================================")
        print("Dataset           :", self.dataset)
        print("Number of items N :", self.N)
        print("Maximum value     :", self.max_value)
        print("Data qubits n     :", self.n)
        print("Quantum states Ne :", self.Ne)
        print("========================================")


    # =============================================================
    # DATA ENCODING
    # =============================================================

    def create_database_state(self):
        """
        Create the equal-amplitude quantum state

             |psi> = 1/sqrt(Ne) sum_x |x>

        over the complete n-qubit search space.

        The actual dataset values are represented by computational
        basis states |d_j>.

        For the Grover minimum search, the oracle decides which
        basis states satisfy x <= current_min.
        """

        amplitudes = np.zeros(self.Ne, dtype=complex)

        # ---------------------------------------------------------
        # Uniform state over the complete search space.
        #
        # This is useful because the paper's experiment uses
        # Ne = 2^n.
        # ---------------------------------------------------------
        amplitudes[:] = 1 / np.sqrt(self.Ne)

        return amplitudes


    # =============================================================
    # STATE PREPARATION
    # =============================================================

    def create_W(self, state):
        """
        Construct W such that

             W |0...0> = |state>

        StatePreparation is used instead of initialize() because
        StatePreparation provides an invertible circuit.
        """

        W = QuantumCircuit(self.n, name="W")

        prep = StatePreparation(state)

        W.append(prep, range(self.n))

        return W


    # =============================================================
    # ORACLE
    # =============================================================

    def create_oracle(self, current_min):
        """
        Oracle for minimum search.

        Mark every basis state |x> satisfying

                    x <= current_min

        with a phase flip.

        The paper describes the oracle mathematically as applying
        a phase to marked states.
        """

        oracle = QuantumCircuit(self.n, name="Oracle")

        # ---------------------------------------------------------
        # Apply a phase of -1 to every state x <= current_min.
        #
        # Qiskit does not provide a direct "phase only on arbitrary
        # basis states" gate, so we construct a diagonal operator.
        # ---------------------------------------------------------

        diagonal = np.ones(self.Ne, dtype=complex)

        for x in range(self.Ne):

            if x <= current_min:
                diagonal[x] = -1

        # Use diagonal gate
        from qiskit.circuit.library import Diagonal

        oracle.append(
            Diagonal(diagonal),
            range(self.n)
        )

        return oracle


    # =============================================================
    # PHASE REVERSAL
    # =============================================================

    def phase_reversal(self, state, phi=np.pi):
        """
        Construct

                    W^dagger I0 W

        which is the phase-reversal/diffusion part of the
        Grover-Long iteration.

        IMPORTANT:
        We use StatePreparation instead of initialize(), because
        StatePreparation is invertible.
        """

        qc = QuantumCircuit(self.n, name="PhaseRev")

        # ---------------------------------------------------------
        # W
        # ---------------------------------------------------------
        prep = StatePreparation(state)

        qc.append(
            prep,
            range(self.n)
        )

        # ---------------------------------------------------------
        # I0
        #
        # Apply phase to |0...0>.
        #
        # To implement a multi-controlled phase, first flip all
        # qubits, perform a multi-controlled Z, then flip them back.
        # ---------------------------------------------------------

        qc.x(range(self.n))

        if self.n == 1:

            qc.z(0)

        elif self.n == 2:

            qc.cz(0, 1)

        else:

            # H + MCX + H gives a multi-controlled Z
            qc.h(self.n - 1)

            qc.mcx(
                list(range(self.n - 1)),
                self.n - 1
            )

            qc.h(self.n - 1)

        qc.x(range(self.n))

        # ---------------------------------------------------------
        # W dagger
        # ---------------------------------------------------------
        qc.append(
            prep.inverse(),
            range(self.n)
        )

        return qc


    # =============================================================
    # GROVER-LONG ITERATION
    # =============================================================

    def grover_long_search(
        self,
        state,
        current_min,
        t_prime
    ):
        """
        Construct the Grover-Long search circuit.

        Structure:

             W
             |
             Oracle
             |
             W^dagger I0 W
             |
             repeat

        The paper's Algorithm 2 uses the Grover-Long operator
        repeatedly.
        """

        qc = QuantumCircuit(
            self.n,
            name=f"GL_{t_prime}"
        )

        prep = StatePreparation(state)

        # ---------------------------------------------------------
        # Initial W
        # ---------------------------------------------------------
        qc.append(
            prep,
            range(self.n)
        )

        # ---------------------------------------------------------
        # Grover-Long iterations
        # ---------------------------------------------------------
        for iteration in range(t_prime):

            # -----------------------------------------------------
            # Oracle
            # -----------------------------------------------------
            oracle = self.create_oracle(current_min)

            qc.compose(
                oracle,
                range(self.n),
                inplace=True
            )

            # -----------------------------------------------------
            # Phase reversal
            #
            # W^dagger I0 W
            #
            # We create this directly.
            # -----------------------------------------------------

            # W dagger
            qc.append(
                prep.inverse(),
                range(self.n)
            )

            # I0
            qc.x(range(self.n))

            if self.n == 1:

                qc.z(0)

            elif self.n == 2:

                qc.cz(0, 1)

            else:

                qc.h(self.n - 1)

                qc.mcx(
                    list(range(self.n - 1)),
                    self.n - 1
                )

                qc.h(self.n - 1)

            qc.x(range(self.n))

            # W
            qc.append(
                prep,
                range(self.n)
            )

        return qc


    # =============================================================
    # MEASUREMENT
    # =============================================================

    def measure_circuit(self, circuit, shots=1):
        """
        Execute a circuit and return the measured integer.
        """

        measure_qc = circuit.copy()

        measure_qc.measure_all()

        compiled = transpile(
            measure_qc,
            self.backend
        )

        result = self.backend.run(
            compiled,
            shots=shots
        ).result()

        counts = result.get_counts()

        measured_bitstring = max(
            counts,
            key=counts.get
        )

        # Qiskit returns classical bits reversed,
        # but int(binary_string, 2) still gives the corresponding
        # computational-basis integer.
        measured_value = int(
            measured_bitstring,
            2
        )

        return measured_value, counts


    # =============================================================
    # CLASSICAL FALLBACK CHECK
    # =============================================================

    def get_candidate_values(self, current_min):
        """
        Values from the actual database satisfying

                    d <= current_min

        This is useful for interpreting the quantum result.
        """

        return [
            d for d in self.dataset
            if d <= current_min
        ]


    # =============================================================
    # OQMSA SEARCH
    # =============================================================

    def search(self, shots=1, draw_circuit=False):
        """
        Execute the OQMSA search.

        Returns:

            minimum
            history
        """

        # ---------------------------------------------------------
        # Reset history
        # ---------------------------------------------------------
        self.history = []

        # ---------------------------------------------------------
        # Random initial candidate d'
        # ---------------------------------------------------------
        current_min = self.rng.choice(
            self.dataset
        )

        print("\n========================================")
        print("STARTING OQMSA")
        print("========================================")

        print(
            "Initial candidate d' :",
            current_min
        )

        # ---------------------------------------------------------
        # Initial quantum state
        # ---------------------------------------------------------
        state = self.create_database_state()

        # ---------------------------------------------------------
        # lambda = 6/5
        #
        # This is the dynamic-growth factor used in Algorithm 1.
        # ---------------------------------------------------------
        lam = 6 / 5

        # ---------------------------------------------------------
        # Search iteration
        # ---------------------------------------------------------
        max_outer_iterations = math.ceil(
            math.log2(self.N)
        ) + 5

        for outer_iteration in range(
            max_outer_iterations
        ):

            # -----------------------------------------------------
            # Estimated number of marked states
            #
            # The paper uses M-hat = d' + 1.
            # -----------------------------------------------------
            M_hat = current_min + 1

            # -----------------------------------------------------
            # t_max
            #
            # We use the standard Grover estimate:
            #
            #     floor(pi/(4*arcsin(sqrt(M/Ne))))
            #
            # -----------------------------------------------------
            ratio = M_hat / self.Ne

            ratio = min(
                max(ratio, 1 / self.Ne),
                1.0
            )

            theta = math.asin(
                math.sqrt(ratio)
            )

            if theta == 0:

                t_max = 1

            else:

                t_max = max(
                    1,
                    math.floor(
                        math.pi / (4 * theta)
                    )
                )

            # -----------------------------------------------------
            # Start t
            # -----------------------------------------------------
            t = 1

            improved = False

            print(
                f"\nOuter iteration {outer_iteration + 1}"
            )

            print(
                "Current d'          :",
                current_min
            )

            print(
                "Estimated M         :",
                M_hat
            )

            print(
                "t_max               :",
                t_max
            )

            # -----------------------------------------------------
            # Dynamic Grover-Long search
            # -----------------------------------------------------
            while (
                t <= t_max
                and current_min > min(self.dataset)
            ):

                # -------------------------------------------------
                # Paper's dynamic strategy:
                #
                # when estimated marked fraction is large,
                # choose a random t'.
                # -------------------------------------------------
                if ratio > 1 / 9:

                    upper = max(
                        0,
                        math.ceil(t)
                    )

                    t_prime = self.rng.randint(
                        0,
                        upper
                    )

                else:

                    t_prime = t_max

                # -------------------------------------------------
                # Do not use zero iterations.
                # -------------------------------------------------
                if t_prime == 0:

                    t_prime = 1

                print(
                    "  t =",
                    t,
                    " t' =",
                    t_prime
                )

                # -------------------------------------------------
                # Construct Grover-Long circuit
                # -------------------------------------------------
                search_circuit = (
                    self.grover_long_search(
                        state,
                        current_min,
                        t_prime
                    )
                )

                # -------------------------------------------------
                # Optional drawing
                # -------------------------------------------------
                if draw_circuit:

                    print(
                        "\nGrover-Long circuit:"
                    )

                    print(
                        search_circuit.draw(
                            "text"
                        )
                    )

                    # Draw only once
                    draw_circuit = False

                # -------------------------------------------------
                # Quantum measurement
                # -------------------------------------------------
                measured_value, counts = (
                    self.measure_circuit(
                        search_circuit,
                        shots=shots
                    )
                )

                # -------------------------------------------------
                # IMPORTANT
                #
                # The measured state is a candidate basis state.
                #
                # Because the quantum search space contains
                # 0 ... Ne-1, a measured value may not correspond
                # to an actual database element.
                #
                # Therefore we only accept it if it occurs in
                # the actual dataset.
                # -------------------------------------------------
                if measured_value in self.dataset:

                    candidate = measured_value

                else:

                    candidate = current_min

                print(
                    "  measured          :",
                    measured_value
                )

                print(
                    "  accepted candidate:",
                    candidate
                )

                # -------------------------------------------------
                # If we found a smaller value
                # -------------------------------------------------
                if candidate < current_min:

                    old_min = current_min

                    current_min = candidate

                    improved = True

                    self.history.append(
                        {
                            "iteration":
                                outer_iteration + 1,

                            "old_min":
                                old_min,

                            "new_min":
                                current_min,

                            "measured":
                                measured_value,

                            "t_prime":
                                t_prime
                        }
                    )

                    print(
                        "  *** NEW MINIMUM ***"
                    )

                    print(
                        "  d' changed:",
                        old_min,
                        "->",
                        current_min
                    )

                    # -------------------------------------------------
                    # Algorithm 1:
                    # if r < d', reset i = 0
                    # -------------------------------------------------
                    outer_iteration = 0

                    break

                # -------------------------------------------------
                # Increase t
                # -------------------------------------------------
                t = t * lam

            # -----------------------------------------------------
            # If current candidate is the actual minimum,
            # we can terminate.
            # -----------------------------------------------------
            if current_min == min(self.dataset):

                break

            # -----------------------------------------------------
            # If no improvement happened, restart with another
            # randomly selected database element.
            # -----------------------------------------------------
            if not improved:

                new_candidate = self.rng.choice(
                    self.dataset
                )

                if new_candidate < current_min:

                    current_min = new_candidate

                    continue

                # If random candidate isn't smaller, use classical
                # candidate selection only to avoid an infinite loop.
                smaller_values = [
                    x for x in self.dataset
                    if x < current_min
                ]

                if smaller_values:

                    current_min = self.rng.choice(
                        smaller_values
                    )

                else:

                    break

        # ---------------------------------------------------------
        # Final result
        # ---------------------------------------------------------
        print("\n========================================")
        print("OQMSA FINISHED")
        print("========================================")

        print(
            "Quantum minimum :",
            current_min
        )

        print(
            "Classical minimum:",
            min(self.dataset)
        )

        print(
            "Correct          :",
            current_min == min(self.dataset)
        )

        return current_min, self.history


    # =============================================================
    # DRAW A GROVER-LONG CIRCUIT
    # =============================================================

    def draw_grover_circuit(
        self,
        current_min=None,
        t_prime=1,
        output="text"
    ):
        """
        Create and draw a Grover-Long circuit.

        output:
            "text" -> ASCII circuit
            "mpl"  -> matplotlib circuit
        """

        if current_min is None:

            current_min = min(self.dataset)

        state = self.create_database_state()

        qc = self.grover_long_search(
            state,
            current_min,
            t_prime
        )

        print(
            f"\nGrover-Long circuit for d' = "
            f"{current_min}, t' = {t_prime}\n"
        )

        if output == "mpl":

            return qc.draw(
                "mpl"
            )

        return qc.draw(
            "text"
        )

In [ ]:
dataset_B = [
    45, 46, 42,
    37, 38, 34,
    21, 22, 18,
    61, 62, 58,
    53, 54, 50,
    5,  6,  2,
    44, 40, 47,
    36, 32, 39,
    20, 16, 23,
    60, 56, 63,
    52, 48, 55,
    4,  7
]

print("N =", len(dataset_B))
print("Minimum classically =", min(dataset_B))

N = 35
Minimum classically = 2


: 

In [ ]:
dataset = [
    45,
    23,
    17,
    31,
    8,
    27,
    12,
    3,
    19,
    7
]

print("Classical minimum =", min(dataset))

oqmsa = OQMSA(
    dataset,
    seed=42
)

minimum, history = oqmsa.search()

print()
print("Classical minimum :", min(dataset))
print("OQMSA minimum     :", minimum)

In [ ]:
import pandas as pd

history_df = pd.DataFrame(history)

print(history_df)